In [ ]:
!pip install bert-extractive-summarizer
!pip install -U sentence-transformers

In [ ]:
import re
from bs4 import BeautifulSoup
import requests
from summarizer.sbert import SBertSummarizer
import pandas as pd
from tqdm import tqdm
import time
tqdm.pandas()

In [ ]:
def get_urls(text):
  r = [item.strip('.').strip('</s>').strip(')').strip(']').strip('.').replace('>)','').replace('>','').replace(',','') for item in re.findall(r'(https?://\S+)', text)]
  if len(r) > 0:
    return r
  else:
    return None

In [ ]:
def get_url_content(url):
  time.sleep(0.5)
  headers = {
    'accept': '*/*',
    'accept-language': 'en-US,en;q=0.9',
    # 'cookie': 'session-id=147-5417338-4035243; session-id-time=2082787201l; ubid-main=133-5931360-4890364; _cc_id=559d3c18e9cefaba2935ac6d99c401db; panoramaId_expiry=1738558256222; panoramaId=457df36569201f5b405a8503278e185ca02cd897b6a27c68b696eef33edcfee8; panoramaIdType=panoDevice; ad-oo=0; ci=e30; session-token=IoGBjzx28L0uuo5oU6kBMchTuRG84qIs1u35A6C2TpB3aqqv3N9zZ931SnCM8pZI5xv+MaYD9ZtjJQSa8iX5QDWQlozY20nhhUYYNDnT7Lu/ZXgUCFrQqyYP2Yenox16Rv91urD6PN60g/Y2ifDhMayrByblLGQ2okad9AaGYEFHJiGdorK3fta4dkOI99eXLLl4YQjIZg7V1qrLFly7ofVY3Ue8Bz/T2nUjNmvckKM6N8Y5iIeGgrCDXIyfJpT0L5XRvIQ2jq0Nqc/X3BnDFvSsxbe4fqoM/3Nc69mwBeH8/YW9SVymx7iMGCVMxRyAvHHX3fXsEtqpycODvFc2HcYaAtsnV+FE; __gads=ID=7cafccf08b1ba769:T=1737172237:RT=1737954898:S=ALNI_Mar8a5qOVejDJphlrEM5suyQjOBJQ; __gpi=UID=00000fcf1e71b35b:T=1737172237:RT=1737954898:S=ALNI_MYUOFhzQr8-gHFS2yKJVmEHITOvQQ; __eoi=ID=c93833b2317531e5:T=1737172237:RT=1737954898:S=AA-AfjbJYR9GuSniUwEkHbgo6JIn; csm-hit=tb:s-J4NNQ33Y4MP7NZN1TR9M|1737955026426&t:1737955026426&adb:adblk_no',
    'priority': 'u=1, i',
    #'referer': 'https://www.imdb.com/name/nm0004418/bio/',
    'sec-ch-ua': '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'}
  if not url or pd.isna(url):
    return None
  try:
    page=requests.get(url, headers=headers)
  except:
    return "page doe not exist!"
  if page.status_code == 200:
    html = page.text
    soup = BeautifulSoup(html, features="html.parser")

    # kill all script and style elements
    for script in soup(["script", "style"]):
        script.extract()    # rip it out

    # get text
    text = soup.get_text()
    #print(text)
    #print("====================================================")
    # break into lines and remove leading and trailing space on each
    lines = (line.strip() for line in text.splitlines())
    try:
      num_lines=len(text.splitlines())
    except:
      num_lines=0
    # break multi-headlines into a line each
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    # drop blank lines
    text = '\n'.join(chunk for chunk in chunks if chunk)
    model = SBertSummarizer('paraphrase-MiniLM-L6-v2')
    if num_lines > 50:
      result = model(text, num_sentences=20)
    else:
      result = text
    return str(result)
  else:
    #print(page.text)
    return "The link is not working "+str(page.status_code)

#importing data

In [ ]:
data=pd.read_excel('data/sample_data_fin_evals_exp.xlsx')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   claim                        203 non-null    object
 1   evidence                     203 non-null    object
 2   label                        203 non-null    object
 3   metadata                     203 non-null    object
 4   explanation                  203 non-null    object
 5   mistral_exp                  203 non-null    object
 6   llama_exp                    203 non-null    object
 7   gpt-4_exp                    203 non-null    object
 8   geval_eval                   203 non-null    object
 9   prom_eval                    203 non-null    object
 10  fine_sure_eval               203 non-null    object
 11  prom_eval_exp                203 non-null    object
 12  prom_eval_mist_exp           203 non-null    object
 13  prom_eval_llama_exp          203 no

In [ ]:
data['links_detected_exp']=data.explanation.progress_apply(get_urls)
data['links_detected_mistral']=data.mistral_exp.progress_apply(get_urls)
data['links_detected_llama']=data.llama_exp.progress_apply(get_urls)
data['links_detected_gpt']=data['gpt-4_exp'].progress_apply(get_urls)

100%|██████████| 203/203 [00:00<00:00, 128268.11it/s]


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   claim                        203 non-null    object
 1   evidence                     203 non-null    object
 2   label                        203 non-null    object
 3   metadata                     203 non-null    object
 4   explanation                  203 non-null    object
 5   mistral_exp                  203 non-null    object
 6   llama_exp                    203 non-null    object
 7   gpt-4_exp                    203 non-null    object
 8   geval_eval                   203 non-null    object
 9   prom_eval                    203 non-null    object
 10  fine_sure_eval               203 non-null    object
 11  prom_eval_exp                203 non-null    object
 12  prom_eval_mist_exp           203 non-null    object
 13  prom_eval_llama_exp          203 no

In [ ]:
url_content_exp, url_content_mistral, url_content_llama, url_content_gpt = [], [], [], []
for i in tqdm(range(len(data))):
  content=''
  links=data.links_detected_exp[i]
  if links and len(links) > 0:
    for url in links:
      content=content+get_url_content(url)+';;;;'
  url_content_exp.append(content)
  content=''
  links=data.links_detected_mistral[i]
  if links and len(links) > 0:
    for url in links:
      content=content+get_url_content(url)+';;;;'
  url_content_mistral.append(content)
  content=''
  links=data.links_detected_llama[i]
  if links and len(links) > 0:
    for url in links:
      content=content+get_url_content(url)+';;;;'
  url_content_llama.append(content)
  content=''
  links=data.links_detected_gpt[i]
  if links and len(links) > 0:
    for url in links:
      content=content+get_url_content(url)+';;;;'
  url_content_gpt.append(content)

  0%|          | 0/203 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
 45%|████▌     | 92/203 [01:08<01:06,  1.67it/s]/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (15) found smaller than n_clusters (16). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
 64%|██████▎   | 129/203 [09:21<42:06, 34.15s/it]/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct cluster

In [ ]:
get_url_content(url)

"She attended St. Joseph's Convent School in Panchgani and was actively involved in extra-curricular activities such as dancing. In her early teens, Kajol was supposed to make her debut as an actress in a movie directed by her mother, Tanuja, but the project was cancelled. She eventually made her acting debut at the age of sixteen with the film Bekhudi (1992), while still in school. Later, she quit school to pursue a full-time career in the film industry. Her film career flourished with commercial successes like Baazigar (1993), Yeh Dillagi (1994), Dilwale Dulhania Le Jayenge (1995), and Kuch Kuch Hota Hai (1998), which established her as a leading star in the 1990s. Another blockbuster Kabhi Khushi Kabhie Gham... (2001) won her multiple awards, after which she took a break. She returned to the industry after a brief period with the romantic thriller Fanaa (2006).She was cast in My Name Is Khan (2010) in 2010 opposite Shah Rukh Khan - it was widely appreciated by critics worldwide. She

In [ ]:
url_content_gpt

[";;;;She attended St. Joseph's Convent School in Panchgani and was actively involved in extra-curricular activities such as dancing. In her early teens, Kajol was supposed to make her debut as an actress in a movie directed by her mother, Tanuja, but the project was cancelled. She eventually made her acting debut at the age of sixteen with the film Bekhudi (1992), while still in school. Later, she quit school to pursue a full-time career in the film industry. Her film career flourished with commercial successes like Baazigar (1993), Yeh Dillagi (1994), Dilwale Dulhania Le Jayenge (1995), and Kuch Kuch Hota Hai (1998), which established her as a leading star in the 1990s. Another blockbuster Kabhi Khushi Kabhie Gham... (2001) won her multiple awards, after which she took a break. She returned to the industry after a brief period with the romantic thriller Fanaa (2006).She was cast in My Name Is Khan (2010) in 2010 opposite Shah Rukh Khan - it was widely appreciated by critics worldwide

In [ ]:
data['url_content_exp']=url_content_exp

In [ ]:
data['url_content_mistral']=url_content_mistral

In [ ]:
data['url_content_llama']=url_content_llama

In [ ]:
data['url_content_gpt']=url_content_gpt

In [ ]:
data.to_excel('sample_data_auto_raters.xlsx', index=False)